# Endgame Vision: Image Classification

This notebook demonstrates Endgame's vision capabilities:
1. Load the CIFAR-10 dataset (real images)
2. Build augmentation pipelines with `AugmentationPipeline`
3. Extract features using `VisionBackbone` (timm)
4. Train a classifier on extracted features
5. Apply test-time augmentation (TTA)
6. Use the high-level `VisionPredictor` API

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

## 1. Load CIFAR-10

CIFAR-10 is a classic image classification benchmark with 60,000 32x32 colour
images across 10 classes. We use a small subset for this demo.

In [ ]:
import torchvision

# CIFAR-10 via torchvision — 60k images, 32x32 colour
train_ds = torchvision.datasets.CIFAR10(root="/tmp/cifar10", train=True, download=True)
test_ds = torchvision.datasets.CIFAR10(root="/tmp/cifar10", train=False, download=True)

# Convert to numpy arrays
images_all = np.concatenate([
    np.array([np.array(img) for img, _ in train_ds]),
    np.array([np.array(img) for img, _ in test_ds]),
], axis=0)
y_all = np.concatenate([np.array(train_ds.targets), np.array(test_ds.targets)])

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]

print(f"Dataset: {images_all.shape[0]} images, {images_all.shape[1]}x{images_all.shape[2]} pixels")
print(f"Classes: {class_names}")
print(f"Label distribution: {np.bincount(y_all)}")

In [ ]:
# Use a 500-sample subset for speed (feature extraction on CPU is slow)
rng = np.random.RandomState(42)
idx = rng.choice(len(images_all), size=500, replace=False)
images = images_all[idx]
labels = y_all[idx]

X_train, X_test, y_train, y_test = train_test_split(
    images, labels, test_size=0.25, random_state=42, stratify=labels,
)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

## 2. Augmentation Pipeline

Endgame wraps albumentations into preset pipelines for common scenarios.
Each preset creates separate train (with augmentation) and val (resize + normalize only) transforms.

In [ ]:
from endgame.vision import AugmentationPipeline

# "standard" preset: horizontal flip, random crop, colour jitter, normalize
aug = AugmentationPipeline(preset="standard", image_size=32, normalize=False)

# Show what transforms are included
print("Training transforms:")
for t in aug.train_transform.transforms:
    print(f"  {t.__class__.__name__}")

print("\nValidation transforms:")
for t in aug.val_transform.transforms:
    print(f"  {t.__class__.__name__}")

In [ ]:
# Apply augmentation to a single image
sample_img = X_train[0]
augmented = aug.train_transform(image=sample_img)["image"]
print(f"Original shape: {sample_img.shape}, Augmented shape: {augmented.shape}")
print(f"Original dtype: {sample_img.dtype}, Augmented dtype: {augmented.dtype}")

In [ ]:
# Mixup and CutMix are available as static methods
mixed, lam = AugmentationPipeline.mixup(X_train[0], X_train[1], alpha=0.4)
print(f"Mixup lambda: {lam:.3f}, shape: {mixed.shape}")

cutmixed, lam = AugmentationPipeline.cutmix(X_train[0], X_train[1], alpha=1.0)
print(f"CutMix lambda: {lam:.3f}, shape: {cutmixed.shape}")

## 3. Feature Extraction with VisionBackbone

`VisionBackbone` wraps any timm model for feature extraction or fine-tuning.
We extract features from a pretrained EfficientNet and train a simple
classifier on top.

In [ ]:
from endgame.vision import VisionBackbone

# List a few available models
models = VisionBackbone.list_models("efficientnet")
print(f"Available EfficientNet variants: {len(models)}")
print(f"First 5: {models[:5]}")

In [ ]:
# Create a backbone for feature extraction (num_classes=None)
backbone = VisionBackbone(
    architecture="efficientnet_b0",
    pretrained=True,
    num_classes=None,  # Feature extraction mode
)

config = backbone.get_config()
print(f"Input size: {config['input_size']}")
print(f"Normalization: mean={config['mean']}, std={config['std']}")

In [ ]:
# Extract features from train and test images
# VisionBackbone handles resizing and normalization internally
print("Extracting train features...", end=" ", flush=True)
train_features = backbone.extract_features(X_train)
print(f"shape: {train_features.shape}")

print("Extracting test features...", end=" ", flush=True)
test_features = backbone.extract_features(X_test)
print(f"shape: {test_features.shape}")

In [ ]:
# Train a LightGBM classifier on the extracted features
import endgame as eg

clf = eg.models.LGBMWrapper(preset="endgame")
clf.fit(train_features, y_train)

y_pred = clf.predict(test_features)
acc = accuracy_score(y_test, y_pred)
print(f"\nAccuracy on CIFAR-10 subset: {acc:.4f}")
print(classification_report(
    y_test, y_pred,
    target_names=class_names,
))

## 4. Test-Time Augmentation (TTA)

TTA applies augmentations at inference time and averages the predictions,
improving robustness without retraining.

In [ ]:
from endgame.vision import TestTimeAugmentation

tta = TestTimeAugmentation(
    augmentations=["identity", "hflip", "vflip"],
    merge_mode="mean",
)
print(f"TTA augmentations: {tta.augmentations}")
print(f"Merge mode: {tta.merge_mode}")

## 5. VisionPredictor (High-Level AutoML API)

For end-to-end image classification, `VisionPredictor` handles
augmentation, backbone selection, training, and TTA automatically.

In [ ]:
from endgame.automl import VisionPredictor

# The VisionPredictor expects a DataFrame with an image column
# containing image paths or numpy arrays, and a label column.
# Here we show the API (training requires GPU for practical use):
print("VisionPredictor parameters:")
print(f"  Default models: efficientnet_b0 (fast), efficientnet_b3 (medium),")
print(f"                  eva02_base (best), convnext_tiny, swin_tiny")
print(f"  Augmentation presets: light, standard, heavy")
print(f"  TTA: identity + hflip by default")

# Example usage (not executed — requires image paths on disk):
# predictor = VisionPredictor(
#     label="species",
#     image_column="image_path",
#     presets="medium_quality",
#     augmentation="standard",
#     use_tta=True,
# )
# predictor.fit(train_df)
# preds = predictor.predict(test_df)

## Summary

In this notebook we:
- Loaded CIFAR-10 from OpenML (real 32x32 colour images)
- Built augmentation pipelines with preset transforms (standard, light, heavy)
- Used Mixup and CutMix for data augmentation
- Extracted features from a pretrained EfficientNet-B0 via `VisionBackbone`
- Trained a LightGBM classifier on the extracted features
- Explored TTA for inference-time robustness
- Reviewed the high-level `VisionPredictor` API

Next steps:
- Try `08_nlp.ipynb` for text classification with transformers
- Try `09_multimodal.ipynb` for combining text, image, and tabular data